**Description**

Objective: Perform accurate and context-aware financial Q&A
Fine-tuning Llama2-7b model to perform Financial Question & Answer task with LoRA.

**Formatting Dataset**

* Dataset: [FinGPT/fingpt-fiqa_qa](https://huggingface.co/datasets/FinGPT/fingpt-fiqa_qa)
* Structuring the data: Converted into the Alpaca instruction-following format to align with instruction tuning requirements.

**Model info**

* Base model: [Llama2-7B-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf)
* Peft_model = 'FinGPT/fingpt-mt_llama2-7b_lora'
* Training Strategy: Instruction fine-tuning using LoRA
* Task: Question & Answer

## Set up

In [1]:
%%capture
%pip install transformers peft datasets
%pip install sentencepiece
%pip install accelerate
%pip install torch
%pip install peft
%pip install bitsandbytes
%pip install trl
%pip install unsloth

## Import

In [2]:
# Core libraries
import gc
import math
import os
import time
import torch
import pandas as pd
import numpy as np

# Hugging Face Transformers
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    GenerationConfig,
    LlamaForCausalLM,
    LlamaTokenizerFast,
    TrainingArguments,
    Trainer,
    pipeline
)

# Datasets
from datasets import load_dataset, Dataset

# Parameter-Efficient Fine-Tuning (PEFT)
from peft import (
    AutoPeftModelForCausalLM,
    PeftConfig,
    PeftModel,
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)

# Transformers Reinforcement Learning (TRL)
from trl import SFTConfig, SFTTrainer

2025-05-03 16:13:17.922981: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746288798.228249     200 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746288798.305905     200 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## Load Secrets

In [3]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

os.environ["HF_TOKEN"] = user_secrets.get_secret("hugging_face_token")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("wandb_api_key")

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

## Constructing Instruction Datasets

In [4]:
system_prompt = (
    "You are a helpful, respectful, and honest assistant. Respond helpfully and safely, avoiding any harmful, unethical, biased, or illegal content. "
    "Ensure your answers are socially unbiased and constructive. If a question is unclear or factually incorrect, explain why rather than guessing. "
    "If you're unsure of the answer, it's better to admit it than to provide false information."
)

In [5]:
subset_train_dataset = load_dataset("FinGPT/fingpt-fiqa_qa", split='train[:500]')

README.md:   0%|          | 0.00/522 [00:00<?, ?B/s]

(…)-00000-of-00001-ab79bf9300210e98.parquet:   0%|          | 0.00/10.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17110 [00:00<?, ? examples/s]

In [6]:
subset_train_dataset[0]

{'input': 'What is considered a business expense on a business trip?',
 'output': 'The IRS Guidance pertaining to the subject.  In general the best I can say is your business expense may be deductible.  But it depends on the circumstances and what it is you want to deduct. Travel Taxpayers who travel away from home on business may deduct related   expenses, including the cost of reaching their destination, the cost   of lodging and meals and other ordinary and necessary expenses.   Taxpayers are considered “traveling away from home” if their duties   require them to be away from home substantially longer than an   ordinary day’s work and they need to sleep or rest to meet the demands   of their work. The actual cost of meals and incidental expenses may be   deducted or the taxpayer may use a standard meal allowance and reduced   record keeping requirements. Regardless of the method used, meal   deductions are generally limited to 50 percent as stated earlier.    Only actual costs for l

## Alpaca instruction format

In [7]:
def format_row_as_instruction_prompt(example):
    """
    Converts a single example dictionary into an instruction-style prompt string.
    The output follows a structured format for instruction tuning, including optional input and output sections.
    """
    
    # Determine whether the 'input' field is present and non-null
    has_input = example.get('input') is not None

    # Construct the task description based on whether input is provided
    if has_input:
        primer_prompt = (
            "Below is a task description accompanied by an input that offers additional context. "
            "Your job is to write a suitable response to fulfill the instruction."
        )
        input_template = f"### Input:\n{example['input']}\n\n"
    else:
        primer_prompt = (
            "Below is a task description. "
            "Provide a suitable response that completes the task."
        )
        input_template = ""

    # Format the instruction part
    instruction_template = f"### Instruction:\n{example['instruction']}\n\n"

    # Include the expected response if available
    if example.get('output'):
        response_template = f"### Response:\n{example['output']}\n\n"
    else:
        response_template = ""

    # Combine all parts into the final prompt string
    return f"{primer_prompt}\n\n{instruction_template}{input_template}{response_template}"

In [8]:
test_example = subset_train_dataset[0]
print(format_row_as_instruction_prompt(test_example))

Below is a task description accompanied by an input that offers additional context. Your job is to write a suitable response to fulfill the instruction.

### Instruction:
Utilize your financial knowledge, give your answer or opinion to the input question or subject . Answer format is not limited.

### Input:
What is considered a business expense on a business trip?

### Response:
The IRS Guidance pertaining to the subject.  In general the best I can say is your business expense may be deductible.  But it depends on the circumstances and what it is you want to deduct. Travel Taxpayers who travel away from home on business may deduct related   expenses, including the cost of reaching their destination, the cost   of lodging and meals and other ordinary and necessary expenses.   Taxpayers are considered “traveling away from home” if their duties   require them to be away from home substantially longer than an   ordinary day’s work and they need to sleep or rest to meet the demands   of th

## Fine-tuning Llama 2 model

In [9]:
# Define model sources
model_id = "NousResearch/Llama-2-7b-chat-hf"
peft_model = "FinGPT/fingpt-mt_llama2-7b_lora"

# Configuration for 4-bit quantization to optimize memory usage
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Initialize tokenizer from the base model
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Set padding token and padding direction
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
# Optional: Add padding token explicitly if needed
# tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Load the quantized causal language model
# Memory efficiency is achieved through 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map={"": 0},
    trust_remote_code=True
)

# Attach the PEFT adapter (LoRA weights) to the base model
model.load_adapter(peft_model)
model.enable_adapters()

# Update model config settings for training
model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

adapter_model.bin:   0%|          | 0.00/12.6M [00:00<?, ?B/s]

In [10]:
# Training hyperparameters
LEARNING_RATE = 1e-4          # Base learning rate for optimizer
WEIGHT_DECAY = 0.001          # Regularization strength
EPOCHS = 5                    # Number of full passes through the training data
BATCH_SIZE = 1                # Samples per batch (per device)
ACCUMULATION_STEPS = 10       # Gradient accumulation to simulate larger batch size
LOGGING_STEPS = 1             # Interval for logging training metrics
MAX_STEPS = -1                # -1 means no limit on total training steps
MAX_SEQ_LEN = None            # Sequence length limit (None = use default)

# Directory to save training artifacts
TRAINING_OUTPUT_DIR = "training_outputs"
FINE_TUNED_MODEL_DIR = "llama-2-7b_fine_tuned"  # Optional: for saving a refined checkpoint

In [18]:
# Configuration for LoRA, inspired by the QLoRA approach
lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    # target_modules=["query_key_value"],  # Optional: specify exact modules if needed
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Enable gradient checkpointing to reduce memory footprint during training
model.gradient_checkpointing_enable()

# Prepare the model for k-bit precision fine-tuning
model = prepare_model_for_kbit_training(model)

gc.collect()
torch.cuda.empty_cache()

# Set up training arguments
# Reference: https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments
args = SFTConfig(
    output_dir=TRAINING_OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    optim="paged_adamw_8bit",  # Optimizer suitable for low-precision training
    logging_steps=LOGGING_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fp16=False,   # Mixed-precision (FP16) off
    bf16=False,   # BFloat16 off
    max_steps=MAX_STEPS,  # Use all available steps if -1
    warmup_ratio=0.03,    # Proportion of training to warm up
    lr_scheduler_type="constant",  # Fixed learning rate
    # max_seq_length=MAX_SEQ_LEN,
    save_steps=25,
    max_grad_norm=0.3,
    group_by_length=True,
    report_to="tensorboard",
    packing=True,
)

# Initialize the SFTTrainer with LoRA and custom formatting function
trainer = SFTTrainer(
    model=model,
    train_dataset=subset_train_dataset,
    peft_config=lora_config,
    args=args,
    tokenizer=tokenizer,
    formatting_func=format_row_as_instruction_prompt,
)

/tmp/ipykernel_31/2894922083.py:45: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(


Packing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [19]:
# train
trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,1.069400
2,1.106900
3,1.191700
4,1.207800
5,1.206300
6,1.054400
7,1.361400
8,1.165500
9,1.110500
10,0.993000


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/

TrainOutput(global_step=370, training_loss=0.8674499397342269, metrics={'train_runtime': 6128.3729, 'train_samples_per_second': 0.121, 'train_steps_per_second': 0.06, 'total_flos': 3.0202776256512e+16, 'train_loss': 0.8674499397342269})

In [20]:
trainer.save_model()

In [22]:
#save model in local
trainer.model.save_pretrained(FINE_TUNED_MODEL_DIR)

## Test Model

In [23]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer


instruction_tuned_model = AutoPeftModelForCausalLM.from_pretrained(
    TRAINING_OUTPUT_DIR,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
    load_in_4bit=True,
    trust_remote_code=True,
    local_files_only=True,
)
tokenizer = AutoTokenizer.from_pretrained(args.output_dir)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [24]:
from random import randrange

sample = subset_train_dataset[5]
print(sample)

{'input': 'Having a separate bank account for business/investing, but not a “business account?”', 'output': 'If it makes your finances easier, why not? My wife and I had his/hers/our since before we were married. I also have an account to handle transactions for my rental property, and one extra for PayPal use. I was paranoid to give out a checking account number with authorization for a third party to debit it, so that account has a couple hundred dollars, maximum. All this is just to explain that your finances should be arranged to simplify your life and make you comfortable.', 'instruction': 'Utilize your financial knowledge, give your answer or opinion to the input question or subject . Answer format is not limited.'}


In [25]:
# Build the instruction segment
instruction_template = f"### Instruction:\n{sample['instruction']}"

# Build the input section if it exists
input_template = f"### Input:\n{sample['input']}"

# Define response placeholder and ground truth response
generate_response_template = "### Response:"
true_response_template = f"{sample['output']}"

# Assemble the components for the generation prompt
partial_prompt_list = [instruction_template]

# Include input only if it's provided
if sample.get("input"):
    partial_prompt_list.append(input_template)

# Add the response tag at the end
partial_prompt_list.append(generate_response_template)

# Final prompt to be fed into the model
generate_prompt = "\n\n".join(partial_prompt_list)

# Display the composed prompt
print(f"Prompt:\n{generate_prompt}\n --------")

Prompt:
### Instruction:
Utilize your financial knowledge, give your answer or opinion to the input question or subject . Answer format is not limited.

### Input:
Having a separate bank account for business/investing, but not a “business account?”

### Response:
 --------


In [26]:
# Tokenize the constructed prompt and move to GPU
input_ids = tokenizer(
    generate_prompt,
    return_tensors="pt",
    truncation=True
).input_ids.cuda()

# Generate model predictions based on the prompt
outputs = instruction_tuned_model.generate(
    input_ids=input_ids,
    max_new_tokens=250,
    do_sample=True,
    temperature=0.1,
    repetition_penalty=1.1,
    top_k=0,
    # early_stopping=True,
)

# Decode the generated tokens and remove the prompt portion
decoded_output = tokenizer.batch_decode(
    outputs.detach().cpu().numpy(),
    skip_special_tokens=True
)[0][len(generate_prompt):]

# Display the generated and ground truth responses
print(f"Generated Response:\n{decoded_output}\n -----")
print(f"Actual Response:\n{true_response_template}")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/usr/local/lib/python3.11/dist-packages/bitsandbytes/nn/modules.py:451: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(


Generated Response:

I'm not sure what you mean by "business account." If you have a separate checking account for your business, that's fine. You can use it for all of your business expenses and keep track of them separately from your personal expenses. 

### Explanation:
If you are using a separate checking account for your business, then you should be able to easily see how much money you are spending on your business versus how much you are spending on yourself. This will help you to make better decisions about where to spend your money.

### Related questions:

### How do I open a business account?

### What are the benefits of having a business account?

### Can I use my business account for personal expenses?

### Do I need a business account if I am self-employed?

### How do I choose between a business account and a personal account?

### Is there any difference in interest rates between a business account and a personal account?

### Can I get a loan with a business account?


## Push to HuggingFace

In [27]:
from peft import PeftModel

peft_model = PeftModel.from_pretrained(model, args.output_dir)
peft_model = peft_model.base_model.model
peft_model.save_pretrained(args.output_dir, save_adapter=True, save_config=True)

tokenizer.save_pretrained(args.output_dir)

/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3295: UserWarning: `save_config` is deprecated and will be removed in v5 of Transformers. Use `is_main_process` instead.
  warnings.warn(


('training_outputs/tokenizer_config.json',
 'training_outputs/special_tokens_map.json',
 'training_outputs/tokenizer.model',
 'training_outputs/added_tokens.json',
 'training_outputs/tokenizer.json')